In [ ]:
"""
====================================================================
SATELLITE IMAGE CLASSIFICATION USING CLASSICAL VISION + ML
====================================================================

PROBLEM STATEMENT
-----------------
Classify satellite images into:
    1. Urban
    2. Natural

using:
    - Handcrafted Image Features
    - Classical Machine Learning Algorithms

This project intentionally avoids deep learning/CNNs
to demonstrate understanding of traditional computer
vision pipelines.

====================================================================
PIPELINE OVERVIEW
====================================================================

    Satellite Image
            ↓
      Preprocessing
            ↓
     Feature Extraction
      ├─ Pixel Features
      ├─ Texture Features
      └─ Gradient Features
            ↓
      Feature Vector
            ↓
     Machine Learning
            ↓
       Prediction

====================================================================
FEATURE TYPES USED
====================================================================

1. PIXEL FEATURES
-----------------
These describe raw image statistics.

Examples:
    - RGB Mean
    - RGB Standard Deviation
    - Color Histograms

Why useful?
------------
Urban images:
    - More gray/concrete colors
    - Less vegetation

Natural images:
    - More green/blue dominance
    - Richer color diversity


2. TEXTURE FEATURES
-------------------
These describe surface patterns.

Examples:
    - GLCM Features
    - Local Binary Patterns (LBP)

Why useful?
------------
Urban areas:
    - Structured repetitive patterns
    - Building textures

Natural areas:
    - Irregular organic textures
    - Forest/terrain randomness


3. GRADIENT FEATURES
--------------------
These describe edges and directional structures.

Examples:
    - HOG Features
    - Edge Density

Why useful?
------------
Urban areas:
    - Roads
    - Buildings
    - Straight edges

Natural areas:
    - Smoother transitions
    - Less structured geometry

====================================================================
"""

# =========================================================
# IMPORT REQUIRED LIBRARIES
# =========================================================

# Operating system utilities
import os

# OpenCV for image processing
import cv2

# Numerical computations
import numpy as np

# Visualization library
import matplotlib.pyplot as plt

# For tracking execution time
import time

# Logging for clean execution monitoring
import logging

# Ignore unnecessary warnings
import warnings

# Progress bar while processing dataset
from tqdm import tqdm

# ---------------------------------------------------------
# SCIKIT-IMAGE FEATURES
# ---------------------------------------------------------

# GLCM = Gray Level Co-occurrence Matrix
# Used for texture analysis
from skimage.feature import (
    graycomatrix,
    graycoprops,

    # Local Binary Pattern
    local_binary_pattern,

    # Histogram of Oriented Gradients
    hog
)

# ---------------------------------------------------------
# SCIKIT-LEARN UTILITIES
# ---------------------------------------------------------

# Dataset splitting
from sklearn.model_selection import train_test_split

# Feature normalization
from sklearn.preprocessing import StandardScaler

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Machine learning models
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


warnings.filterwarnings("ignore")


# =========================================================
# CONFIGURATION CLASS
# =========================================================

class Config:
    """
    Centralized configuration class.

    Storing parameters in one place makes the code:
        - Cleaner
        - Easier to modify
        - More maintainable
    """

    # -----------------------------------------------------
    # DATASET LOCATION
    # -----------------------------------------------------

    # Path where EuroSAT dataset is stored
    DATASET_PATH = "EuroSAT"

    # -----------------------------------------------------
    # IMAGE SIZE
    # -----------------------------------------------------

    # All images are resized to same dimensions.
    #
    # Why?
    # ----
    # Machine learning models require fixed-size features.
    #
    # Larger images:
    #     + More detail
    #     - Slower computation
    #
    # Smaller images:
    #     + Faster
    #     - May lose details
    #
    IMAGE_SIZE = (128, 128)

    # -----------------------------------------------------
    # CLASS MAPPING
    # -----------------------------------------------------

    # These EuroSAT classes are considered URBAN
    URBAN_CLASSES = [
        "Residential",
        "Industrial",
        "Highway"
    ]

    # These classes are considered NATURAL
    NATURAL_CLASSES = [
        "Forest",
        "River",
        "Pasture",
        "SeaLake",
        "AnnualCrop",
        "PermanentCrop",
        "HerbaceousVegetation"
    ]

    # -----------------------------------------------------
    # DATASET LIMIT
    # -----------------------------------------------------

    # Optional limit for faster experimentation.
    #
    # Can be increased later for better accuracy.
    #
    MAX_IMAGES_PER_CLASS = 1000

    # -----------------------------------------------------
    # TRAIN TEST SPLIT
    # -----------------------------------------------------

    TEST_SIZE = 0.2

    # Fixed random state ensures reproducibility.
    RANDOM_STATE = 42

    # -----------------------------------------------------
    # LOCAL BINARY PATTERN PARAMETERS
    # -----------------------------------------------------

    # Radius around central pixel
    LBP_RADIUS = 2

    # Number of neighboring sample points
    LBP_POINTS = 16

    # -----------------------------------------------------
    # HOG PARAMETERS
    # -----------------------------------------------------

    # Number of gradient directions
    HOG_ORIENTATIONS = 9

    # Size of cell
    HOG_PIXELS_PER_CELL = (8, 8)

    # Number of cells grouped together
    HOG_CELLS_PER_BLOCK = (2, 2)

    # -----------------------------------------------------
    # LOGGING LEVEL
    # -----------------------------------------------------

    LOG_LEVEL = logging.INFO


# =========================================================
# LOGGER SETUP
# =========================================================

"""
Logging helps monitor:
    - Progress
    - Errors
    - Timing
    - Accuracy

Useful for debugging and project presentation.
"""

logging.basicConfig(
    level=Config.LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

logger = logging.getLogger(__name__)


# =========================================================
# TIMER DECORATOR
# =========================================================

def timer(func):
    """
    Decorator function to measure execution time.

    Useful for:
        - Performance analysis
        - Optimization
        - Understanding bottlenecks
    """

    def wrapper(*args, **kwargs):

        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()

        logger.info(
            f"{func.__name__} completed "
            f"in {end - start:.2f} sec"
        )

        return result

    return wrapper


# =========================================================
# IMAGE PREPROCESSING
# =========================================================

def preprocess_image(image_path):
    """
    Load and preprocess image.

    Steps:
    ------
    1. Read image
    2. Convert BGR → RGB
    3. Resize image
    4. Convert to grayscale

    Why grayscale?
    ---------------
    Many texture and gradient algorithms
    work better on intensity information.
    """

    # Read image using OpenCV
    image = cv2.imread(image_path)

    # Safety check
    if image is None:
        raise ValueError(
            f"Could not load image: {image_path}"
        )

    # OpenCV loads as BGR by default.
    # Convert to RGB for consistency.
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    # Resize image
    image = cv2.resize(
        image,
        Config.IMAGE_SIZE
    )

    # Convert to grayscale
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2GRAY
    )

    return image, gray


# =========================================================
# PIXEL-LEVEL FEATURES
# =========================================================

def extract_pixel_features(image):
    """
    Extract raw pixel statistical features.

    FEATURES:
    ---------
    1. RGB Mean
    2. RGB Standard Deviation
    3. Color Histograms

    WHY IMPORTANT?
    --------------
    Urban:
        - Gray buildings
        - Roads
        - Less vegetation

    Natural:
        - More greens/blues
        - Greater color diversity
    """

    features = []

    # -----------------------------------------------------
    # RGB MEAN
    # -----------------------------------------------------

    # Average intensity for R,G,B channels
    mean_rgb = np.mean(
        image,
        axis=(0, 1)
    )

    # -----------------------------------------------------
    # RGB STANDARD DEVIATION
    # -----------------------------------------------------

    # Measures color spread/variation
    std_rgb = np.std(
        image,
        axis=(0, 1)
    )

    # -----------------------------------------------------
    # COLOR HISTOGRAMS
    # -----------------------------------------------------

    """
    Histogram represents distribution of colors.

    32 bins used for each channel.

    Why normalization?
    ------------------
    Makes histograms independent of image size.
    """

    hist_features = []

    for channel in range(3):

        hist = cv2.calcHist(
            [image],
            [channel],
            None,
            [32],
            [0, 256]
        )

        # Normalize histogram
        hist = cv2.normalize(
            hist,
            hist
        ).flatten()

        hist_features.extend(hist)

    # Combine all pixel features
    features.extend(mean_rgb)
    features.extend(std_rgb)
    features.extend(hist_features)

    return np.array(features)


# =========================================================
# GLCM TEXTURE FEATURES
# =========================================================

def extract_glcm_features(gray):
    """
    Extract GLCM texture features.

    GLCM = Gray Level Co-occurrence Matrix

    It measures:
        How often pixel intensities occur together.

    VERY IMPORTANT for texture analysis.

    FEATURES:
    ---------
    1. Contrast
    2. Homogeneity
    3. Energy
    4. Correlation
    """

    # Create GLCM matrix
    glcm = graycomatrix(
        gray,

        # Distance between compared pixels
        distances=[1],

        # Horizontal comparison
        angles=[0],

        # Grayscale intensity levels
        levels=256,

        symmetric=True,
        normed=True
    )

    # -----------------------------------------------------
    # TEXTURE PROPERTIES
    # -----------------------------------------------------

    contrast = graycoprops(
        glcm,
        'contrast'
    )[0, 0]

    homogeneity = graycoprops(
        glcm,
        'homogeneity'
    )[0, 0]

    energy = graycoprops(
        glcm,
        'energy'
    )[0, 0]

    correlation = graycoprops(
        glcm,
        'correlation'
    )[0, 0]

    return np.array([
        contrast,
        homogeneity,
        energy,
        correlation
    ])


# =========================================================
# LOCAL BINARY PATTERN FEATURES
# =========================================================

def extract_lbp_features(gray):
    """
    Extract Local Binary Pattern features.

    LBP captures:
        Local texture micro-patterns.

    HOW IT WORKS:
    -------------
    Compare neighboring pixels with center pixel.

    Result:
        Binary pattern representing local texture.

    WHY USEFUL?
    ------------
    Urban:
        Structured repetitive patterns

    Natural:
        Organic irregular patterns
    """

    lbp = local_binary_pattern(
        gray,
        Config.LBP_POINTS,
        Config.LBP_RADIUS,
        method="uniform"
    )

    # Create histogram of LBP patterns
    hist, _ = np.histogram(
        lbp.ravel(),

        bins=np.arange(
            0,
            Config.LBP_POINTS + 3
        ),

        range=(
            0,
            Config.LBP_POINTS + 2
        )
    )

    # Convert to float
    hist = hist.astype("float")

    # Normalize histogram
    hist /= (hist.sum() + 1e-6)

    return hist


# =========================================================
# HOG FEATURES
# =========================================================

def extract_hog_features(gray):
    """
    Extract Histogram of Oriented Gradients features.

    HOG captures:
        Edge directions and structural geometry.

    VERY IMPORTANT FOR:
        - Roads
        - Buildings
        - Urban layouts

    WHY?
    ----
    Urban scenes contain:
        - Many straight lines
        - Strong directional edges

    Natural scenes:
        - More chaotic gradients
    """

    features = hog(
        gray,

        orientations=Config.HOG_ORIENTATIONS,

        pixels_per_cell=Config.HOG_PIXELS_PER_CELL,

        cells_per_block=Config.HOG_CELLS_PER_BLOCK,

        block_norm='L2-Hys',

        visualize=False,

        feature_vector=True
    )

    return features


# =========================================================
# EDGE DENSITY FEATURE
# =========================================================

def extract_edge_density(gray):
    """
    Compute edge density.

    Edge density =
        Number of edge pixels / Total pixels

    WHY USEFUL?
    ------------
    Urban images:
        High edge density
        due to buildings and roads.

    Natural images:
        Lower edge density.
    """

    # Detect edges using Canny detector
    edges = cv2.Canny(
        gray,
        100,
        200
    )

    # Compute edge ratio
    edge_density = (
        np.sum(edges > 0) / edges.size
    )

    return np.array([edge_density])


# =========================================================
# COMBINE ALL FEATURES
# =========================================================

def extract_all_features(image, gray):
    """
    Combine ALL handcrafted features
    into ONE feature vector.

    FINAL VECTOR CONTAINS:
    ----------------------
    1. Pixel Features
    2. GLCM Features
    3. LBP Features
    4. HOG Features
    5. Edge Density
    """

    pixel_features = extract_pixel_features(image)

    glcm_features = extract_glcm_features(gray)

    lbp_features = extract_lbp_features(gray)

    hog_features = extract_hog_features(gray)

    edge_density = extract_edge_density(gray)

    # Concatenate all features
    combined = np.concatenate([
        pixel_features,
        glcm_features,
        lbp_features,
        hog_features,
        edge_density
    ])

    return combined


# =========================================================
# DATASET LOADING
# =========================================================

@timer
def load_dataset():
    """
    Load entire dataset and extract features.

    OUTPUT:
    -------
    X = Feature vectors
    y = Labels

    Label Mapping:
        Urban  -> 1
        Natural -> 0
    """

    X = []
    y = []

    logger.info("Loading dataset...")

    class_dirs = os.listdir(
        Config.DATASET_PATH
    )

    for class_name in class_dirs:

        class_path = os.path.join(
            Config.DATASET_PATH,
            class_name
        )

        if not os.path.isdir(class_path):
            continue

        # -------------------------------------------------
        # LABEL ASSIGNMENT
        # -------------------------------------------------

        if class_name in Config.URBAN_CLASSES:
            label = 1

        elif class_name in Config.NATURAL_CLASSES:
            label = 0

        else:
            continue

        image_files = os.listdir(class_path)

        image_files = image_files[
            :Config.MAX_IMAGES_PER_CLASS
        ]

        logger.info(
            f"Processing {class_name}"
        )

        for file in tqdm(image_files):

            try:

                image_path = os.path.join(
                    class_path,
                    file
                )

                image, gray = preprocess_image(
                    image_path
                )

                features = extract_all_features(
                    image,
                    gray
                )

                X.append(features)

                y.append(label)

            except Exception as e:

                logger.error(
                    f"Error: {file} -> {e}"
                )

    X = np.array(X)
    y = np.array(y)

    logger.info(
        f"Final Dataset Shape: {X.shape}"
    )

    return X, y


# =========================================================
# MODEL TRAINING
# =========================================================

@timer
def train_models(X_train, y_train):
    """
    Train multiple classical ML models.

    Models Used:
    ------------
    1. Logistic Regression
       Simple baseline model.

    2. Random Forest
       Ensemble-based robust classifier.

    3. Support Vector Machine
       Very effective for handcrafted features.
    """

    models = {

        "LogisticRegression":
            LogisticRegression(
                max_iter=1000
            ),

        "RandomForest":
            RandomForestClassifier(
                n_estimators=100,
                random_state=Config.RANDOM_STATE,
                n_jobs=-1
            ),

        "SVM":
            SVC(
                kernel='rbf',
                probability=True
            )
    }

    trained_models = {}

    for name, model in models.items():

        logger.info(
            f"Training {name}"
        )

        model.fit(
            X_train,
            y_train
        )

        trained_models[name] = model

    return trained_models


# =========================================================
# MODEL EVALUATION
# =========================================================

def evaluate_model(model, X_test, y_test, name):
    """
    Evaluate model performance.

    Metrics Used:
    --------------
    1. Accuracy
    2. Precision
    3. Recall
    4. F1 Score
    5. Confusion Matrix
    """

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    logger.info(
        f"{name} Accuracy: {accuracy:.4f}"
    )

    print("\n")
    print("=" * 60)
    print(f"{name} RESULTS")
    print("=" * 60)

    print(
        classification_report(
            y_test,
            predictions,
            target_names=[
                "Natural",
                "Urban"
            ]
        )
    )

    print("Confusion Matrix:\n")

    print(
        confusion_matrix(
            y_test,
            predictions
        )
    )


# =========================================================
# FEATURE VISUALIZATION
# =========================================================

def visualize_features(image_path):
    """
    Visualize extracted feature representations.

    Visualization includes:
        1. Original Image
        2. LBP Texture Map
        3. Edge Map
        4. HOG Visualization

    Useful for:
        - Understanding feature extraction
        - Demonstrating results to professor
    """

    image, gray = preprocess_image(
        image_path
    )

    # -----------------------------------------------------
    # LBP MAP
    # -----------------------------------------------------

    lbp = local_binary_pattern(
        gray,
        Config.LBP_POINTS,
        Config.LBP_RADIUS,
        method="uniform"
    )

    # -----------------------------------------------------
    # EDGE MAP
    # -----------------------------------------------------

    edges = cv2.Canny(
        gray,
        100,
        200
    )

    # -----------------------------------------------------
    # HOG VISUALIZATION
    # -----------------------------------------------------

    _, hog_image = hog(
        gray,

        orientations=Config.HOG_ORIENTATIONS,

        pixels_per_cell=Config.HOG_PIXELS_PER_CELL,

        cells_per_block=Config.HOG_CELLS_PER_BLOCK,

        visualize=True
    )

    # -----------------------------------------------------
    # DISPLAY RESULTS
    # -----------------------------------------------------

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(18, 5)
    )

    axes[0].imshow(image)
    axes[0].set_title("Original")

    axes[1].imshow(lbp, cmap='gray')
    axes[1].set_title("LBP")

    axes[2].imshow(edges, cmap='gray')
    axes[2].set_title("Edges")

    axes[3].imshow(hog_image, cmap='gray')
    axes[3].set_title("HOG")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()

    plt.show()


# =========================================================
# EXTERNAL IMAGE TESTING
# =========================================================

def predict_external_image(
    model,
    scaler,
    image_path
):
    """
    Predict a completely unseen external image.

    This demonstrates:
        - Generalization ability
        - Real-world robustness
    """

    image, gray = preprocess_image(
        image_path
    )

    features = extract_all_features(
        image,
        gray
    )

    # Scaling must match training data scaling
    features = scaler.transform([features])

    prediction = model.predict(features)[0]

    probability = model.predict_proba(features)[0]

    label = (
        "Urban"
        if prediction == 1
        else "Natural"
    )

    logger.info(
        f"Prediction: {label}"
    )

    logger.info(
        f"Confidence: {np.max(probability):.4f}"
    )

    # Display result
    plt.figure(figsize=(6, 6))

    plt.imshow(image)

    plt.title(
        f"{label} "
        f"({np.max(probability):.2%})"
    )

    plt.axis("off")

    plt.show()


# =========================================================
# MAIN PIPELINE
# =========================================================

@timer
def main():
    """
    Main execution pipeline.
    """

    # -----------------------------------------------------
    # LOAD DATASET
    # -----------------------------------------------------

    X, y = load_dataset()

    # -----------------------------------------------------
    # TRAIN TEST SPLIT
    # -----------------------------------------------------

    """
    Why split dataset?
    ------------------
    Train Set:
        Used for learning.

    Test Set:
        Used for evaluation.

    Prevents overfitting evaluation.
    """

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,

            test_size=Config.TEST_SIZE,

            random_state=Config.RANDOM_STATE,

            stratify=y
        )
    )

    logger.info(
        f"Train Shape: {X_train.shape}"
    )

    logger.info(
        f"Test Shape: {X_test.shape}"
    )

    # -----------------------------------------------------
    # FEATURE SCALING
    # -----------------------------------------------------

    """
    Why scaling?
    -------------
    Features have different ranges.

    Example:
        Histogram → small values
        HOG → large values

    StandardScaler normalizes feature ranges,
    improving ML performance.
    """

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )

    # -----------------------------------------------------
    # TRAIN MODELS
    # -----------------------------------------------------

    models = train_models(
        X_train,
        y_train
    )

    # -----------------------------------------------------
    # EVALUATE MODELS
    # -----------------------------------------------------

    for name, model in models.items():

        evaluate_model(
            model,
            X_test,
            y_test,
            name
        )

    # -----------------------------------------------------
    # VISUALIZE FEATURES
    # -----------------------------------------------------

    sample_image = os.path.join(
        Config.DATASET_PATH,
        "Residential",

        os.listdir(
            os.path.join(
                Config.DATASET_PATH,
                "Residential"
            )
        )[0]
    )

    visualize_features(sample_image)

    # -----------------------------------------------------
    # EXTERNAL IMAGE TEST
    # -----------------------------------------------------

    """
    Place any external satellite image as:
        external_test.jpg

    in the project directory.
    """

    external_image = "external_test.jpg"

    if os.path.exists(external_image):

        # SVM usually performs best
        best_model = models["SVM"]

        predict_external_image(
            best_model,
            scaler,
            external_image
        )

    else:

        logger.warning(
            "No external image found."
        )


# =========================================================
# PROGRAM ENTRY POINT
# =========================================================

"""
Python starts execution here.
"""

if __name__ == "__main__":

    main()